In [1]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine,URL

In [2]:
# connect to DB
conn = psycopg2.connect(
  host='localhost',
  port=5432,
  dbname='postgres',
  user='postgres',
  password='postgres'
)
conn.set_session(autocommit=True)
cursor = conn.cursor()

In [3]:
engine = create_engine(
  URL.create(
    drivername='postgresql+psycopg2',
    host='localhost',
    port=5432,
    database='postgres',
    username='postgres',
    password='postgres'
  )
)

In [4]:
# read m1_total
cursor.execute(
  'select * from m1_total'
)
m1_total_df = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

---
x 구성

In [6]:
# 거주인구 - 반경 500m 내
cursor.execute(
  f'''
  select
    pnu,
    sum(pop.pop_cnt) live_pop
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m1_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (
    select
      pop_cnt,
      geom_3857
    from live_pop
  ) as pop
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        500
      ),
      pop.geom_3857
    )
  group by 1
  '''
)
lot_live_pop = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [7]:
# 직장인구 - 반경 1000m 내
cursor.execute(
  f'''
  select
    pnu,
    sum(pop.pop_cnt) work_pop
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m1_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (
    select
      pop_cnt,
      geom_3857
    from work_pop
  ) as pop
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        1000
      ),
      pop.geom_3857
    )
  group by 1
  '''
)
lot_work_pop = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [8]:
# 인근 학교 개수 - 1. 어린이집 / 500m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_1_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m1_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '어린이집') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        500
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_1_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [9]:
# 인근 학교 개수 - 2. 유치원 / 500m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_2_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m1_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '유치원') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        500
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_2_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [10]:
# 인근 학교 개수 - 3. 초등학교 / 1000m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_3_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m1_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '초등학교') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        1000
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_3_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [11]:
# 인근 학교 개수 - 4. 중학교 / 1000m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_4_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m1_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '중학교') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        1000
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_4_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [12]:
# 인근 학교 개수 - 5. 고등학교 / 1000m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_5_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m1_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '고등학교') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        1000
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_5_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

In [13]:
# 인근 학교 개수 - 6. 대학교 / 1000m
cursor.execute(
  f'''
  select
    pnu,
    count(*) school_type_6_cnt
  from (
    select
      y.pnu,
      lot_polygon.geom_3857
    from (
      select
        distinct pnu
      from m1_y
    ) as y
    left join lot_polygon
    on y.pnu = lot_polygon.pnu
  ) y,
  (select geom_3857 from school where school_type = '대학교') as sch
  where
    st_intersects(
      st_buffer(
        y.geom_3857,
        1000
      ),
      sch.geom_3857
    )
  group by 1
  '''
)
school_type_6_cnt = pd.DataFrame(
  cursor.fetchall(),
  columns=[col.name for col in cursor.description]
)

---
m4_y + x 통합

In [27]:
m4_total_df = m1_total_df.merge(
  lot_live_pop,
  how='left',
  on='pnu'
).merge(
  lot_work_pop,
  how='left',
  on='pnu'
).merge(
  school_type_1_cnt,
  how='left',
  on='pnu'
).merge(
  school_type_2_cnt,
  how='left',
  on='pnu'
).merge(
  school_type_3_cnt,
  how='left',
  on='pnu'
).merge(
  school_type_4_cnt,
  how='left',
  on='pnu'
).merge(
  school_type_5_cnt,
  how='left',
  on='pnu'
).merge(
  school_type_6_cnt,
  how='left',
  on='pnu'
)

In [28]:
m4_total_df.to_sql(
  'm4_total',
  engine,
  if_exists='replace',
  index=False
)

210

---
전달용 데이터 저장하기

In [29]:
m4_total_df.to_csv(
  'm4_data_06.23.csv',
  sep=',',
  index=False
)